In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [6]:
# Set paths
raw_path = Path("d:/bluestock_mf_capstone/data/raw")
processed_path = Path("d:/bluestock_mf_capstone/data/processed")
db_path = Path("d:/bluestock_mf_capstone/data/db")

# Create directories if they don't exist
processed_path.mkdir(parents=True, exist_ok=True)
db_path.mkdir(parents=True, exist_ok=True)

print(f"Raw path: {raw_path.absolute()}")
print(f"Processed path: {processed_path.absolute()}")
print(f"DB path: {db_path.absolute()}")

Raw path: d:\bluestock_mf_capstone\data\raw
Processed path: d:\bluestock_mf_capstone\data\processed
DB path: d:\bluestock_mf_capstone\data\db


In [7]:
# List all files in raw folder
csv_files = list(raw_path.glob("*.*"))
print("Files found:")
for f in csv_files:
    print(f"  - {f.name}")

Files found:
  - 01_fund_master.csv
  - 02_nav_history.csv
  - 03_aum_by_fund_house.csv
  - 04_monthly_sip_inflows.csv
  - 05_category_inflows.csv
  - 06_industry_folio_count.csv
  - 07_scheme_performance.csv
  - 08_investor_transactions.csv
  - 09_portfolio_holdings.csv
  - 10_benchmark_indices.csv


In [10]:
def load_and_analyze(file_path):
    """Load CSV/Excel and print analysis"""
    
    # Determine file type
    if file_path.suffix.lower() == '.csv':
        df = pd.read_csv(file_path)
    elif file_path.suffix.lower() in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path)
    else:
        print(f"Unknown format: {file_path.suffix}")
        return None
    
    print(f"\n{'='*60}")
    print(f"FILE: {file_path.name}")
    print(f"{'='*60}")
    
    # Shape
    print(f"📊 SHAPE: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # Data types
    print(f"\n📝 DATA TYPES:")
    for col, dtype in df.dtypes.items():
        print(f"   {col}: {dtype}")
    
    # First 3 rows
    print(f"\n👀 FIRST 3 ROWS:")
    print(df.head(3).to_string())
    
    # Missing values
    missing = df.isnull().sum()
    missing_cols = missing[missing > 0]
    if len(missing_cols) > 0:
        print(f"\n⚠️ MISSING VALUES:")
        for col, count in missing_cols.items():
            print(f"   {col}: {count} missing ({count/len(df)*100:.1f}%)")
    else:
        print(f"\n✅ No missing values")
    
    # Duplicates
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        print(f"\n⚠️ DUPLICATES: {dup_count} duplicate rows")
    else:
        print(f"\n✅ No duplicate rows")
    
    return df

In [11]:
# Load all files
data_dict = {}

for file_path in csv_files:
    df = load_and_analyze(file_path)
    if df is not None:
        # Store using file name without extension as key
        key = file_path.stem
        data_dict[key] = df


FILE: 01_fund_master.csv
📊 SHAPE: 40 rows, 15 columns

📝 DATA TYPES:
   amfi_code: int64
   fund_house: object
   scheme_name: object
   category: object
   sub_category: object
   plan: object
   launch_date: object
   benchmark: object
   expense_ratio_pct: float64
   exit_load_pct: float64
   min_sip_amount: int64
   min_lumpsum_amount: int64
   fund_manager: object
   risk_category: object
   sebi_category_code: object

👀 FIRST 3 ROWS:
   amfi_code       fund_house                                 scheme_name category sub_category     plan launch_date             benchmark  expense_ratio_pct  exit_load_pct  min_sip_amount  min_lumpsum_amount   fund_manager risk_category sebi_category_code
0     119551  SBI Mutual Fund   SBI Bluechip Fund - Regular Plan - Growth   Equity    Large Cap  Regular  2006-02-14         NIFTY 100 TRI               1.54            1.0             500                1000  Sohini Andani      Moderate               EC01
1     119552  SBI Mutual Fund    SBI Blue

In [12]:
print("\n" + "="*60)
print("ANOMALIES SUMMARY")
print("="*60)

anomalies = []

for name, df in data_dict.items():
    # Check for constant columns (all same value)
    constant_cols = [col for col in df.columns if df[col].nunique() == 1]
    if constant_cols:
        anomalies.append(f"{name}: Constant columns - {constant_cols}")
    
    # Check for columns that are entirely null
    all_null_cols = [col for col in df.columns if df[col].isnull().all()]
    if all_null_cols:
        anomalies.append(f"{name}: All-null columns - {all_null_cols}")
    
    # Check for negative values in numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        if (df[col] < 0).any():
            neg_count = (df[col] < 0).sum()
            anomalies.append(f"{name}: Negative values in '{col}' ({neg_count} rows)")
    
    # Check for date columns that might be misformatted (optional)
    for col in df.columns:
        if 'date' in col.lower() and df[col].dtype == 'object':
            # Try to see if it's a date string
            try:
                pd.to_datetime(df[col], errors='raise')
            except:
                anomalies.append(f"{name}: Column '{col}' looks like date but not standard format")

if anomalies:
    print("\n⚠️ Anomalies found:")
    for a in anomalies:
        print(f"   • {a}")
else:
    print("\n✅ No major anomalies detected")


ANOMALIES SUMMARY

⚠️ Anomalies found:
   • 01_fund_master: Constant columns - ['min_sip_amount']
   • 07_scheme_performance: Negative values in 'max_drawdown_pct' (40 rows)
   • 09_portfolio_holdings: Constant columns - ['portfolio_date']


In [13]:
# Create summary dataframe
summary_data = []
for name, df in data_dict.items():
    summary_data.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing_Values": df.isnull().sum().sum(),
        "Duplicate_Rows": df.duplicated().sum()
    })

summary_df = pd.DataFrame(summary_data)

# Save to processed folder
summary_df.to_csv(processed_path / "ingestion_summary.csv", index=False)

print("\n✅ Summary saved to:", processed_path / "ingestion_summary.csv")
print("\n📊 INGESTION SUMMARY:")
print(summary_df.to_string())


✅ Summary saved to: d:\bluestock_mf_capstone\data\processed\ingestion_summary.csv

📊 INGESTION SUMMARY:
                    Dataset   Rows  Columns  Missing_Values  Duplicate_Rows
0            01_fund_master     40       15               0               0
1            02_nav_history  46000        3               0               0
2      03_aum_by_fund_house     90        5               0               0
3    04_monthly_sip_inflows     48        6              12               0
4       05_category_inflows    144        3               0               0
5   06_industry_folio_count     21        6               0               0
6     07_scheme_performance     40       19               0               0
7  08_investor_transactions  32778       13               0               0
8     09_portfolio_holdings    322        8               0               0
9      10_benchmark_indices   8050        3               0               0


In [14]:
# ============================================
# FUND MASTER EXPLORATION
# ============================================

print("="*60)
print("FUND MASTER EXPLORATION")
print("="*60)

# Load fund master (already in data_dict from earlier)
fund_master = data_dict.get('01_fund_master')
if fund_master is None:
    # Fallback: load directly
    import pandas as pd
    from pathlib import Path
    raw_path = Path("data/raw")
    fund_path = list(raw_path.glob("*fund_master*"))[0]
    if fund_path.suffix == '.csv':
        fund_master = pd.read_csv(fund_path)
    else:
        fund_master = pd.read_excel(fund_path)

print(f"\n📊 Shape: {fund_master.shape}")
print(f"📋 Columns: {list(fund_master.columns)}\n")

# Find relevant columns dynamically
for col in fund_master.columns:
    col_lower = col.lower()
    if 'fund' in col_lower or 'amc' in col_lower:
        print(f"🏢 UNIQUE FUND HOUSES ('{col}'):")
        unique_vals = fund_master[col].dropna().unique()
        print(f"   Count: {len(unique_vals)}")
        print(f"   Examples: {list(unique_vals)[:10]}\n")
        
    elif 'category' in col_lower and 'sub' not in col_lower:
        print(f"📂 UNIQUE CATEGORIES ('{col}'):")
        unique_vals = fund_master[col].dropna().unique()
        print(f"   {list(unique_vals)}\n")
        
    elif 'sub' in col_lower and 'category' in col_lower:
        print(f"📁 UNIQUE SUB-CATEGORIES ('{col}'):")
        unique_vals = fund_master[col].dropna().unique()
        print(f"   Count: {len(unique_vals)}")
        print(f"   Examples: {list(unique_vals)[:15]}\n")
        
    elif 'risk' in col_lower:
        print(f"⚠️ UNIQUE RISK GRADES ('{col}'):")
        unique_vals = fund_master[col].dropna().unique()
        print(f"   {list(unique_vals)}\n")
        
    elif 'code' in col_lower:
        print(f"🔢 AMFI SCHEME CODE ('{col}'):")
        print(f"   Min: {fund_master[col].min()}, Max: {fund_master[col].max()}")
        print(f"   Sample codes: {fund_master[col].dropna().head(10).tolist()}\n")

print("✅ Fund master exploration complete.")

FUND MASTER EXPLORATION

📊 Shape: (40, 15)
📋 Columns: ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']

🔢 AMFI SCHEME CODE ('amfi_code'):
   Min: 100016, Max: 149324
   Sample codes: [119551, 119552, 119598, 119599, 119120, 100016, 125497, 100033, 125498, 100025]

🏢 UNIQUE FUND HOUSES ('fund_house'):
   Count: 10
   Examples: ['SBI Mutual Fund', 'HDFC Mutual Fund', 'ICICI Prudential MF', 'Nippon India MF', 'Kotak Mahindra MF', 'Axis Mutual Fund', 'Aditya Birla Sun Life MF', 'UTI Mutual Fund', 'Mirae Asset MF', 'DSP Mutual Fund']

📂 UNIQUE CATEGORIES ('category'):
   ['Equity', 'Debt']

📁 UNIQUE SUB-CATEGORIES ('sub_category'):
   Count: 12
   Examples: ['Large Cap', 'Small Cap', 'Gilt', 'Mid Cap', 'Short Duration', 'Value', 'Liquid', 'Index/ETF', 'Flexi Cap', 'Index', 'Large & Mid Cap', 'ELSS']

🏢 

In [15]:
# ============================================
# AMFI CODE VALIDATION
# ============================================

print("="*60)
print("AMFI CODE VALIDATION")
print("="*60)

# Load nav_history (already in data_dict)
nav_history = data_dict.get('02_nav_history')
if nav_history is None:
    nav_path = list(Path("data/raw").glob("*nav_history*"))[0]
    if nav_path.suffix == '.csv':
        nav_history = pd.read_csv(nav_path)
    else:
        nav_history = pd.read_excel(nav_path)

# Identify code columns
fund_code_col = None
for col in fund_master.columns:
    if 'code' in col.lower():
        fund_code_col = col
        break

nav_code_col = None
for col in nav_history.columns:
    if 'code' in col.lower():
        nav_code_col = col
        break

print(f"Fund Master code column: '{fund_code_col}'")
print(f"NAV History code column: '{nav_code_col}'")

# Get unique codes as strings
fund_codes = set(fund_master[fund_code_col].dropna().astype(str))
nav_codes = set(nav_history[nav_code_col].dropna().astype(str))

print(f"\n📊 Unique codes in Fund Master: {len(fund_codes)}")
print(f"📊 Unique codes in NAV History: {len(nav_codes)}")

# Find missing and extra
missing_in_nav = fund_codes - nav_codes
extra_in_nav = nav_codes - fund_codes

print("\n" + "="*60)
print("VALIDATION RESULTS")
print("="*60)

if len(missing_in_nav) == 0:
    print("✅ PASS: Every code in fund_master exists in nav_history")
else:
    print(f"⚠️ {len(missing_in_nav)} codes in fund_master are missing from nav_history")
    print(f"   Missing codes (first 10): {list(missing_in_nav)[:10]}")

if len(extra_in_nav) > 0:
    print(f"\nℹ️ {len(extra_in_nav)} codes in nav_history are not in fund_master")
    print("   (These may be closed/merged or historical schemes)")

# Coverage percentage
coverage = (len(fund_codes - missing_in_nav) / len(fund_codes)) * 100 if len(fund_codes) > 0 else 0
print(f"\n📈 Code coverage: {coverage:.2f}%")

# Data quality summary
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)
print(f"Fund Master rows: {len(fund_master)}")
print(f"NAV History rows: {len(nav_history)}")
print(f"Missing codes: {len(missing_in_nav)} ({100-coverage:.2f}% of fund codes)")
print(f"Extra codes: {len(extra_in_nav)}")

if coverage == 100:
    print("Grade: A+ (Complete coverage)")
elif coverage >= 95:
    print("Grade: A (Excellent)")
elif coverage >= 85:
    print("Grade: B (Good)")
else:
    print("Grade: C (Needs investigation)")

# Save report
from pathlib import Path
report_path = Path("data/processed") / "amfi_validation_report.txt"
report_path.parent.mkdir(parents=True, exist_ok=True)

report = f"""AMFI CODE VALIDATION REPORT
=================================
Generated: {pd.Timestamp.now()}

Fund Master rows: {len(fund_master)}
NAV History rows: {len(nav_history)}

Unique Fund Codes: {len(fund_codes)}
Unique NAV Codes: {len(nav_codes)}

Missing in NAV: {len(missing_in_nav)} ({100-coverage:.2f}%)
Extra in NAV: {len(extra_in_nav)}

Code Coverage: {coverage:.2f}%

Recommendation: Use INNER JOIN on matching codes for analysis.
"""

with open(report_path, 'w') as f:
    f.write(report)
print(f"\n✅ Report saved to: {report_path}")

AMFI CODE VALIDATION
Fund Master code column: 'amfi_code'
NAV History code column: 'amfi_code'

📊 Unique codes in Fund Master: 40
📊 Unique codes in NAV History: 40

VALIDATION RESULTS
✅ PASS: Every code in fund_master exists in nav_history

📈 Code coverage: 100.00%

DATA QUALITY SUMMARY
Fund Master rows: 40
NAV History rows: 46000
Missing codes: 0 (0.00% of fund codes)
Extra codes: 0
Grade: A+ (Complete coverage)

✅ Report saved to: data\processed\amfi_validation_report.txt
